# quantum-cortex — the control plus a local convolution at 3000 steps, 8 pairs, eight seeds (register Rev60, declared before this run)

One variable added to the ladder's transformer control: L3's depthwise causal convolution (kernel 3) on the attention's normalised input in every block (`attn_local_conv`, transparent to every recorded hash at its default). Eight units at the same steps, seeds, tier and evaluation as the references of RESULTS row 30, written into `metrics/mqar/ref3000-ckpt/`; then the declared readout, paired on seed: the convolved control against the control, against L3 pure, and against the full accuracy of `RB-bare` and `RB-critical` (row 31).

**GPU T4, Internet On, about 25 minutes** (the control's units took about 2 min 30 each in stage A). **Secrets:** `GITHUB_TOKEN`; `KAGGLE_USERNAME` and `KAGGLE_KEY` for the relay (optional). Cells: 1 this header; 2 the clone; 3 the relay; 4 the resume lookup; 5 the eight units; 6 the readout; 7 the files to commit. Run 2, 3 and 4 alone first and paste their outputs; then Save & Run All. **Download the log as a file before reading any number.**


In [ ]:
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GITHUB_TOKEN")
!rm -rf quantum-cortex
!git clone https://{token}@github.com/vaneeckhoutnicolas/quantum-cortex.git
%cd quantum-cortex
!pip -q install numpy jsonschema


In [ ]:
# the checkpoint relay (optional): every finished unit pushed to the private dataset, pulled back after a relaunch
import os
from kaggle_secrets import UserSecretsClient
try:
    os.environ['KAGGLE_USERNAME'] = UserSecretsClient().get_secret('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = UserSecretsClient().get_secret('KAGGLE_KEY')
    from cortex_data import relay
    print('relay:', 'on' if relay.available() else 'off (kaggle command missing)')
except Exception:
    print('relay: off (secrets KAGGLE_USERNAME / KAGGLE_KEY not attached)')


In [ ]:
# Previous ref3000 directories mounted as Inputs (resume). The stage A references and the sigma units are in the clone itself.
import glob
PREV_REF = sorted(set(glob.glob('/kaggle/input/**/ref3000-ckpt', recursive=True)))
print('resume references from:', PREV_REF or '(none — fresh start)')


In [ ]:
# the eight units of the control plus convolution, at the references' steps, seeds, tier and evaluation (row 30)
resume = ' '.join(f'--resume-from {p}' for p in ['metrics/mqar/sigma-l3-3000-ckpt'] + PREV_REF)
!python -m cortex_eval.resumable_ladder --paths ARCH-none+local-conv --steps 3000 --seeds 1337 2024 7 42 99 3 11 2026 --tiers 8x128 --ckpt-dir metrics/mqar/ref3000-ckpt --time-budget-min 100 --relay quantum-cortex-relay-ref3000 {resume}


In [ ]:
# the declared readout (Rev60): no training; paired on seed against the control, L3 pure and the two arms of row 31
!python -m cortex_eval.resumable_rb --control-conv-readout metrics/mqar/ref3000-ckpt/READOUT-control-conv-3000steps.json --ckpt-dir metrics/mqar/rb3000-ckpt --reference-from metrics/mqar/ref3000-ckpt metrics/mqar/sigma-l3-3000-ckpt --arms RB-bare RB-critical --seeds 1337 2024 7 42 99 3 11 2026 --tiers 8x128


In [ ]:
import glob
print('\n'.join(sorted(glob.glob('metrics/mqar/ref3000-ckpt/unit-ARCH-none+local-conv*.json') + glob.glob('metrics/mqar/ref3000-ckpt/*ARCH-none*local-conv*.json'))))
print('Commit the new unit files, the SUBSET file and the READOUT file of metrics/mqar/ref3000-ckpt/ — the units are the record.')
